## Chat API : LangChain

เมื่อใช้งานผ่าน interface ของ LangChain จะสามารถทำได้โดยขั้นตอนเหล่านี้

In [1]:
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.109.1
    Uninstalling openai-1.109.1:
      Successfully uninstalled openai-1.109.1


In [28]:
!pip -q install langchain==0.0.232

In [5]:
import openai
from google.colab import userdata
import os
openai.api_key = userdata.get("OPENAI_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_KEY')

# LangChain: Agents

Agent เป็นหนึ่งใน Experiment ที่โด่งดังที่สุด และติดเทรนด์ใยช่วงนี้ เครื่องมือหลายๆอย่าง เช่น AutoGPT, babyAGI ล้วนมีลักษณะคล้ายคลึงกับ Agent

โดยสิ่งที่ Agent สามารถทำได้คือ 1. รับคำสั่ง 2. วางแผน 3. ปฎิบัติตามคำสั่ง 4. เรียนรู้ เพื่อที่นำไปปฎิบัติต่อไป

เมื่อมีโครงสร้างพื้นฐานดังกล่าวแล้ว Agent จึงเป็นมากกว่า Chatbot ธรรมดาที่เพียงแค่ตอบคำถามจากสิ่งที่เราต้องการ แต่มันสามารถสังเกต วางแผน และเรียนรู้ที่จะทำงานให้ออกมาดีที่สุด

สิ่งหนึ่งที่ทำให้ความสามารถของ Agent มีสูงมาก และประสิทธิภาพสูงมาก คือ Tools หรือเครื่องมือต่าง ๆ ที่ Agent สามารถใช้ได้ ไม่ว่าจะเป็นการใช้งาน Search Engine (DuckDuckGo) หรืออ่านข้อความจาก Wikipedia หรือแม้แต่ Tools ที่เราต้องการสร้างเองก็ได้

## เนื้อหา:

* ใช้งาน Built-in tools ใน LangChain : DuckDuckGo search, Wikipedia
* สร้าง Tools เอง

In [39]:
!pip -q install wikipedia

In [41]:
from langchain.agents.agent_toolkits import create_python_agent
from langchain.agents import load_tools, initialize_agent
from langchain.agents import AgentType
from langchain.tools.python.tool import PythonREPLTool
from langchain.python import PythonREPL
from langchain.chat_models import ChatOpenAI

In [45]:
llm = ChatOpenAI(temperature=0)

In [47]:
tools = load_tools(["llm-math","wikipedia"], llm=llm)

In [48]:
agent= initialize_agent(
    tools,
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose = True)

In [49]:
agent("What is the 25% of 300?")



> Entering new AgentExecutor chain...
Thought: We can use a calculator to find 25% of 300. 
Action:
```
{
  "action": "Calculator",
  "action_input": "25% of 300"
}
```
Observation: Answer: 75.0
Thought:Final Answer: 75.0

> Finished chain.


{'input': 'What is the 25% of 300?', 'output': '75.0'}

In [50]:
question = "Tom M. Mitchell is an American computer scientist and the Founders University Professor at Carnegie Mello University (CMU) what book did he write?"
result   = agent(question)



> Entering new AgentExecutor chain...
Thought: I should use Wikipedia to find out which book Tom M. Mitchell wrote.
Action:
```
{
  "action": "Wikipedia",
  "action_input": "Tom M. Mitchell"
}
```

/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')



Observation: Page: Tom M. Mitchell
Summary: Tom Michael Mitchell (born August 9, 1951) is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). He is a founder and former chair of the Machine Learning Department at CMU. Mitchell is known for his contributions to the advancement of machine learning, artificial intelligence, and cognitive neuroscience and is the author of the textbook Machine Learning. He is a member of the United States National Academy of Engineering since 2010. He is also a Fellow of the American Academy of Arts and Sciences, the American Association for the Advancement of Science and a Fellow and past president of the Association for the Advancement of Artificial Intelligence. In October 2018, Mitchell was appointed as the Interim Dean of the School of Computer Science at Carnegie Mellon.

Page: Tom Mitchell (Australian footballer)
Summary: Thomas Mitchell (born 31 May 1993) is a professional Australian rules footb

## Python Agent

หนึ่งใน Agent ที่ทรงพลังที่สุด เนื่องจากมันสามารถ Execute code ได้

In [51]:
agent = create_python_agent(
    llm,
    tool=PythonREPLTool(),
    verbose = True
)

In [52]:
customer_list = [["Harrison","Chase"],
                 ["Lang","Chain"],
                 ["Dolly","Too"],
                 ["Elle","Elem"],
                 ["Geoff","Fusion"],
                 ["Trance","Former"],
                 ["Jen","Ayai"]]

In [54]:
agent.run(f"""
Sort these customers by last name and then first name and print the output : {customer_list}
""")



> Entering new AgentExecutor chain...
We can use the sorted() function in Python to sort the list of customers by last name first and then by first name.

Action: Python_REPL
Action Input: customers = [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]
sorted_customers = sorted(customers, key=lambda x: (x[1], x[0]))
print(sorted_customers)
Observation: [['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]

Thought:The customers have been sorted by last name and then by first name.

Final Answer: [['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]

> Finished chain.


"[['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]"

#### ดูการทำงานของ Agent

In [53]:
import langchain

In [55]:
langchain.debug=True

In [56]:
agent.run(f"""Sort these customers by last name and then first name and print the output: {customer_list}""")
langchain.debug=False

[chain/start] [1:RunTypeEnum.chain:AgentExecutor] Entering Chain run with input:
{
  "input": "Sort these customers by last name and then first name and print the output: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]"
}
[chain/start] [1:RunTypeEnum.chain:AgentExecutor > 2:RunTypeEnum.chain:LLMChain] Entering Chain run with input:
{
  "input": "Sort these customers by last name and then first name and print the output: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]",
  "agent_scratchpad": "",
  "stop": [
    "\nObservation:",
    "\n\tObservation:"
  ]
}
[llm/start] [1:RunTypeEnum.chain:AgentExecutor > 2:RunTypeEnum.chain:LLMChain > 3:RunTypeEnum.llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: You are an agent designed to write and execute python code to answer questions.\nYou ha

## สร้าง tool แบบ Custom

In [57]:
from langchain.agents import tool
from datetime import date

In [58]:
@tool
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

In [59]:
agent= initialize_agent(
    tools + [time],
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose = True)

In [60]:
try:
    result = agent("whats the date today?")
except:
    print("exception on external access")



> Entering new AgentExecutor chain...
Thought: I should use the `time` tool to find out today's date.

Action:
```
{
  "action": "time",
  "action_input": ""
}
```


Observation: 2025-10-08
Thought:Final Answer: 2025-10-08

> Finished chain.
